In [244]:
library(ArchR)
library(Seurat)
library(SeuratObject)
library(dplyr)
library(Cairo)
library(schard)
.libPaths()

sessionInfo()

setwd("~/workspace/reha/archr_Fhl2/")
addArchRThreads(threads = 16) 
addArchRGenome("mm10")

[1] "/home/jinba/anaconda3/envs/ArchR/lib/R/library"

R version 4.1.2 (2021-11-01)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /home/jinba/anaconda3/envs/ArchR/lib/libopenblasp-r0.3.27.so

Random number generation:
 RNG:     L'Ecuyer-CMRG 
 Normal:  Inversion 
 Sample:  Rejection 
 
locale:
 [1] LC_CTYPE=ja_JP.utf8       LC_NUMERIC=C             
 [3] LC_TIME=ja_JP.utf8        LC_COLLATE=ja_JP.utf8    
 [5] LC_MONETARY=ja_JP.utf8    LC_MESSAGES=ja_JP.utf8   
 [7] LC_PAPER=ja_JP.utf8       LC_NAME=C                
 [9] LC_ADDRESS=C              LC_TELEPHONE=C           
[11] LC_MEASUREMENT=ja_JP.utf8 LC_IDENTIFICATION=C      

attached base packages:
 [1] parallel  stats4    grid      stats     graphics  grDevices utils    
 [8] datasets  methods   base     

other attached packages:
 [1] ggridges_0.5.7              VennDiagram_1.8.2          
 [3] futile.logger_1.4.9         presto_1.0.0               
 [5] enrichR_3.4                 ggrepel_0.9.6              
 [7] 

Setting default number of Parallel threads to 16.

Setting default genome to Mm10.



In [245]:
library(showtext)
showtext_auto()

In [246]:
CM_RNA <- schard::h5ad2seurat("~/workspace/reha/data/adata/adata_Fhl2_count.h5ad")

archr_project_path = "~/workspace/reha/archr_Fhl2/CM_atac"

Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from rna to rna_”
Warning message:
“Invalid name supplied, making object name syntactically valid. New object name is X_indexsampletypen_genesn_genes_by_countstotal_countstotal_counts_mtpct_counts_mtdoublet_scorepredicted_doubletleidencell_typebatch; see ?make.names for more details on syntax validity”


In [ ]:
CM_RNA <- CM_RNA %>%
  NormalizeData() %>%
  FindVariableFeatures(nfeatures = 5000) %>%
  ScaleData(vars.to.regress = c("percent.mt","nCount_RNA")) %>%
  RunPCA(npcs = 30, verbose = FALSE)
CM_RNA <- CM_RNA %>%
  RunUMAP(reduction = "pca", dims = 1:20) %>%
  FindNeighbors(reduction = "pca", dims = 1:20) %>%
  FindClusters(resolution = 0.1)
p <- DimPlot(CM_RNA, group.by='leiden', label=TRUE) 
p

In [ ]:
CM_RNA@assays$RNA@data

In [ ]:
# prepare CM subset -------------------------------------------------------

projReha <- readRDS("./rds/projReha.rds")
cluster_info <- getCellColData(projReha, select = "Clusters")
head(cluster_info)
subset_clusters <- c("C1","C2","C3")
cluster_info$cellname <- row.names(cluster_info)
cellname_cm <- cluster_info[cluster_info$Clusters %in% subset_clusters,]$cellname

proj <- subsetArchRProject(
  ArchRProj = projReha,
  cells = cellname_cm,
  outputDirectory = "./CM_atac",
  dropCells = TRUE,force = TRUE
)
setwd("./CM_atac")

In [ ]:
# dimension reduction -------------------------------------------------------
proj <- addIterativeLSI(
  ArchRProj = proj,
  useMatrix = "TileMatrix",
  name = "IterativeLSI",
  iterations = 2,
  clusterParams = list(
    resolution = c(0.2, 0.3),
    sampleCells = 10000,
    n.start = 10
  ),
  varFeatures = 15000,
  dimsToUse = 1:30, 
  force=TRUE
)

In [ ]:
proj <- addUMAP(
  ArchRProj = proj,
  reducedDims = "IterativeLSI",
  name = "UMAP",
  nNeighbors = 20,
  spread = 0.5,
  minDist = 1,
  metric = "cosine", 
  force = TRUE
)

In [ ]:
proj <- addClusters(input = proj, reducedDims = "IterativeLSI", force = TRUE, res=0.5)

In [13]:
plotEmbedding(
  ArchRProj = proj,
  colorBy = "cellColData",
  name = "Sample",
  embedding = "UMAP"
)

ArchR logging to : ArchRLogs/ArchR-plotEmbedding-9a09d3c79f21c-Date-2026-03-03_Time-15-56-02.log
If there is an issue, please report to github with logFile!

Getting UMAP Embedding

ColorBy = cellColData

Plotting Embedding

1 


ArchR logging successful to : ArchRLogs/ArchR-plotEmbedding-9a09d3c79f21c-Date-2026-03-03_Time-15-56-02.log



In [14]:
plotEmbedding(
  ArchRProj = proj,
  colorBy = "cellColData",
  name = "Type",
  embedding = "UMAP"
)

ArchR logging to : ArchRLogs/ArchR-plotEmbedding-9a09d662098e0-Date-2026-03-03_Time-15-56-16.log
If there is an issue, please report to github with logFile!

Getting UMAP Embedding

ColorBy = cellColData

Plotting Embedding

1 


ArchR logging successful to : ArchRLogs/ArchR-plotEmbedding-9a09d662098e0-Date-2026-03-03_Time-15-56-16.log



In [199]:
# まず実際のラベル文字列を確認（ここがズレてると絶対反映されません）
unique(getCellColData(proj, select = "Type")$Type)

p <- plotEmbedding(
  ArchRProj = proj,
  colorBy   = "cellColData",
  name      = "Type",
  embedding = "UMAP",
  pal = c(
    "Ex"     = "#FF8C00",  # orange
    "Fhl2OE" = "#d62728",  # green
    "SED"    = "#2E8B57"   # blue
  ),
  labelMeans = FALSE
)

p

[1] "SED"    "Ex"     "Fhl2OE"

ArchR logging to : ArchRLogs/ArchR-plotEmbedding-cc2f745540978-Date-2026-03-09_Time-15-12-32.log
If there is an issue, please report to github with logFile!

Getting UMAP Embedding

ColorBy = cellColData

Plotting Embedding

1 


ArchR logging successful to : ArchRLogs/ArchR-plotEmbedding-cc2f745540978-Date-2026-03-09_Time-15-12-32.log



In [200]:
p <- plotEmbedding(
  ArchRProj = proj,
  colorBy   = "cellColData",
  name      = "Type",
  embedding = "UMAP",
  pal = c(
    "Ex"     = "#FF8C00",
    "Fhl2OE" = "#d62728",
    "SED"    = "#2E8B57"
  ),
  labelMeans = FALSE
) +
  theme(
    axis.title = element_text(size = 18),   # x,y ラベル
    axis.text  = element_text(size = 16)    # 目盛り
  )

p

ArchR logging to : ArchRLogs/ArchR-plotEmbedding-cc2f777993307-Date-2026-03-09_Time-15-13-09.log
If there is an issue, please report to github with logFile!

Getting UMAP Embedding

ColorBy = cellColData

Plotting Embedding

1 


ArchR logging successful to : ArchRLogs/ArchR-plotEmbedding-cc2f777993307-Date-2026-03-09_Time-15-13-09.log



In [ ]:
# 結果の確認
p1 <- plotEmbedding(
  ArchRProj = proj,
  colorBy = "cellColData",
  name = "Sample",
  embedding = "UMAP"
)
p2 <- plotEmbedding(
  ArchRProj = proj,
  colorBy = "cellColData",
  name = "Clusters",
  embedding = "UMAP"
)
p1+p2

In [ ]:
markerGenes = c("Fhl2","Pecam1","Ttn")

In [ ]:
plotEmbedding(
  ArchRProj = proj, 
  colorBy = "GeneScoreMatrix", 
  name = markerGenes, 
  continuousSet = "horizonExtra",
  embedding = "UMAP",
  imputeWeights = getImputeWeights(proj)
)

In [ ]:
# intergration with snRNAseq ------------------------------------------------------------
library(raster)
CM_RNA$BioClassification <- CM_RNA$type
CM_RNA$group <- CM_RNA$type
CM_RNA <- UpdateSeuratObject(CM_RNA)

In [ ]:
# seurat 4.4.1 seurat object 4.1.4 works
proj <- addGeneIntegrationMatrix(
  ArchRProj = proj, 
  useMatrix = "GeneScoreMatrix",
  matrixName = "GeneIntegrationMatrix",
  reducedDims = "IterativeLSI",
  seRNA = CM_RNA,
  groupRNA = "BioClassification",
  nameCell = "predictedCell_Un",
  nameGroup = "predictedGroup_Un",
  nameScore = "predictedScore_Un",
  force = TRUE
)
proj

In [ ]:
# add gene matrix
proj <- addImputeWeights(proj)
getAvailableMatrices(proj) # gene score matrix is added

In [ ]:
plotEmbedding(ArchRProj = proj, colorBy = "cellColData", name = "predictedGroup_Un", embedding = "UMAP", labelMeans=FALSE)

In [ ]:
p2 <- plotEmbedding(
  ArchRProj = proj, 
  colorBy = "GeneIntegrationMatrix", 
  name = markerGenes, 
  continuousSet = "horizonExtra",
  embedding = "UMAP",
  imputeWeights = getImputeWeights(proj)
)
p3 <- plotEmbedding(
  ArchRProj = proj, 
  colorBy = "GeneScoreMatrix", 
  name = markerGenes, 
  continuousSet = "horizonExtra",
  embedding = "UMAP",
  imputeWeights = getImputeWeights(proj)
)

In [ ]:
proj <- saveArchRProject(ArchRProj = proj)

saveRDS(proj,"./CM_atac_projReha.rds")

In [247]:
setwd("~/workspace/reha/archr_Fhl2/CM_atac/")
proj <- readRDS("./CM_atac_projReha.rds")

# Peak calling

In [ ]:
table(proj$Sample,proj$predictedGroup_Un)

In [ ]:
proj$Type <- substr(proj$Sample, 1, nchar(proj$Sample) - 1)

In [ ]:
plotEmbedding(
  ArchRProj = proj,
  colorBy = "cellColData",
  name = "Type",
  embedding = "UMAP"
)

In [ ]:
pal <- c(
  Ex = "#1f78b4",      # 青
  Fhl2OE = "#33a02c",  # 緑
  SED = "#e31a1c"      # 赤
)

p <- plotEmbedding(
  ArchRProj = proj,
  colorBy = "cellColData",
  name = "Type",
  embedding = "UMAP",
  pal = pal
)


p + theme(
  legend.text = element_text(size = 18),
  legend.title = element_text(size = 16)
)


In [ ]:
# make pseudo bulk ------------------------------------------------------------
library(BSgenome.Mmusculus.UCSC.mm10)
proj <- addGroupCoverages(proj, groupBy="Type", force = TRUE)

In [ ]:
conflicts(detail = TRUE) 

In [ ]:
detach("package:raster", unload = TRUE)

In [ ]:
# Call peaks --------------------------------------------------------------
library(GenomicRanges)


#MACS2
pathToMacs2 <- "/home/jinba/anaconda3/envs/ArchR/bin/macs2"
#pathToMacs2 <- findMacs2()
proj <- addReproduciblePeakSet(
  ArchRProj = proj, 
  groupBy = "Type", 
  pathToMacs2 = pathToMacs2
)
proj <- addPeakMatrix(proj)

In [ ]:
getPeakSet(proj)
getAvailableMatrices(proj)

In [ ]:
#  Identifying Marker Peaks  ----------------------------------------------
markersPeaks <- getMarkerFeatures(
  ArchRProj = proj, 
  useMatrix = "PeakMatrix", 
  groupBy = "Type",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)


In [ ]:
#get markers for cluster
markerList <- getMarkers(markersPeaks, cutOff = "FDR <= 0.1 & Log2FC >= 0.1")
heatmapPeaks <- plotMarkerHeatmap(
  seMarker = markersPeaks, 
  cutOff = "FDR <= 0.1 & Log2FC >= 0.1",
  transpose = TRUE,
  plotLog2FC = TRUE
)

In [ ]:
heatmapPeaks
plotPDF(heatmapPeaks, name = "Peak-Marker-Heatmap", width = 8, height = 6, ArchRProj = proj, addDOC = FALSE)
draw(heatmapPeaks, heatmap_legend_side = "bot", annotation_legend_side = "bot")

In [ ]:
#MA plots
pma <- markerPlot(seMarker = markersPeaks, name = "Fhl2OE", cutOff = "FDR <= 0.1 & Log2FC >= 0.05", plotAs = "MA")
pma

In [ ]:
#Browser track
p <- plotBrowserTrack(
  ArchRProj = proj, 
  groupBy = "Type", 
  geneSymbol = c("Ppargc1a"),
  features =  getMarkers(markersPeaks, cutOff = "FDR <= 0.1 & Log2FC >= 0.05", returnGR = TRUE)["Fhl2OE"],
  upstream = 100000,
  downstream = 200000
)
grid::grid.newpage()
grid::grid.draw(p$Ppargc1a)

In [ ]:
proj <- addMotifAnnotations(ArchRProj = proj, motifSet = "cisbp", name = "Motif", force = TRUE)

In [ ]:
proj@peakAnnotation

In [ ]:
## Marker peak motief -----------------------------------------------------------------------
enrichMotifs <- peakAnnoEnrichment(
  seMarker = markersPeaks,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC >= 0.05"
  )
enrichMotifs
heatmapEM <- plotEnrichHeatmap(enrichMotifs, n = 800, transpose = TRUE)
heatmapEM


In [ ]:
ComplexHeatmap::draw(heatmapEM, heatmap_legend_side = "bot", annotation_legend_side = "bot")

In [ ]:
# Pairwise testing for marker peaks ---------------------------------------
markerTest_Ex <- getMarkerFeatures(
  ArchRProj = proj, 
  useMatrix = "PeakMatrix",
  groupBy = "Type",
  testMethod = "wilcoxon",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  useGroups = "Ex",
  bgdGroups = "SED"
)

In [ ]:
markerTest_Fhl2 <- getMarkerFeatures(
  ArchRProj = proj, 
  useMatrix = "PeakMatrix",
  groupBy = "Type",
  testMethod = "wilcoxon",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  useGroups = "Fhl2OE",
  bgdGroups = "SED"
)

In [ ]:
markerTest_Fhl2vsEx <- getMarkerFeatures(
  ArchRProj = proj, 
  useMatrix = "PeakMatrix",
  groupBy = "Type",
  testMethod = "wilcoxon",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  useGroups = "Fhl2OE",
  bgdGroups = "Ex"
)

In [ ]:
## Motif enrichment analysis -------------------------------------------------------
proj <- addMotifAnnotations(ArchRProj = proj, motifSet = "cisbp", name = "Motif", force = TRUE)
motifsUp_Ex <- peakAnnoEnrichment(
  seMarker = markerTest_Ex,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC >= 0.1")
motifsUp_Fhl2 <- peakAnnoEnrichment(
  seMarker = markerTest_Fhl2,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC >= 0.1")

In [ ]:
## Motif enrichment analysis -------------------------------------------------------
proj <- addMotifAnnotations(ArchRProj = proj, motifSet = "cisbp", name = "Motif", force = TRUE)
motifsDown_Ex <- peakAnnoEnrichment(
  seMarker = markerTest_Ex,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC <= -0.1")
motifsDown_Fhl2 <- peakAnnoEnrichment(
  seMarker = markerTest_Fhl2,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC <= -0.1")

In [ ]:
## Motif enrichment analysis -------------------------------------------------------
proj <- addMotifAnnotations(ArchRProj = proj, motifSet = "cisbp", name = "Motif", force = TRUE)
motifsUp_Fhl2vsEx <- peakAnnoEnrichment(
  seMarker = markerTest_Fhl2vsEx,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC >= 0.1")
motifsDown_Fhl2vsEx <- peakAnnoEnrichment(
  seMarker = markerTest_Fhl2vsEx,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC <= -0.1")

In [ ]:
# Rank plot ---------------------------------------------------------------
rankplot <- function(motifsUp,name){
  df <- data.frame(TF = rownames(motifsUp), mlog10Padj = assay(motifsUp)[,1])
  df <- df[order(df$mlog10Padj, decreasing = TRUE),]
  df$rank <- seq_len(nrow(df))
  ggUp <- ggplot(df, aes(rank, mlog10Padj, color = mlog10Padj)) + 
    geom_point(size = 1) +
    ggrepel::geom_label_repel(
    data = df[rev(seq_len(30)), ], aes(x = rank, y = mlog10Padj, label = TF), 
    size = 1.5,
    nudge_x = 2,
    color = "black"
    ) + theme_ArchR() + 
   ylab("-log10(P-adj) Motif Enrichment") + 
    xlab("Rank Sorted TFs Enriched") +
    scale_color_gradientn(colors = paletteContinuous(set = "comet"))
  print(ggUp)
  ggsave(paste0("./",name,"_upTF.tiff"), plot = ggUp, width = 5, height = 8, units = "in")
}

In [ ]:
rankplot(motifsUp_Ex,"ExvsSED_up")
rankplot(motifsDown_Ex,"ExvsSED_Down")

In [ ]:
rankplot(motifsUp_Fhl2,"Fhl2vsSED_up")
rankplot(motifsDown_Fhl2,"Fhl2vsSED_Down")

In [ ]:
df <- data.frame(TF = rownames(motifsUp_Fhl2), mlog10Padj = assay(motifsUp_Fhl2)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),]
df$rank <- seq_len(nrow(df))

In [ ]:
df

In [ ]:
df <- data.frame(TF = rownames(motifsDown_Fhl2), mlog10Padj = assay(motifsDown_Fhl2)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),]
df$rank <- seq_len(nrow(df))

In [ ]:
df

In [ ]:
df <- data.frame(TF = rownames(motifsDown_Ex), mlog10Padj = assay(motifsDown_Ex)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),]
df$rank <- seq_len(nrow(df))

In [ ]:
df

In [ ]:
df <- data.frame(TF = rownames(motifsUp_Ex), mlog10Padj = assay(motifsUp_Ex)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),]
df$rank <- seq_len(nrow(df))

In [ ]:
df

In [ ]:
df <- data.frame(TF = rownames(motifsUp_Fhl2vsEx), mlog10Padj = assay(motifsUp_Fhl2vsEx)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),]
df$rank <- seq_len(nrow(df))

In [ ]:
df

In [ ]:
df <- data.frame(TF = rownames(motifsDown_Fhl2vsEx), mlog10Padj = assay(motifsDown_Fhl2vsEx)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),]
df$rank <- seq_len(nrow(df))

In [ ]:
df

In [ ]:
## motif
enrichMotifs <- peakAnnoEnrichment(
  seMarker = markersPeaks,
  ArchRProj = proj,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC >= 0.1"
)

heatmapEM <- plotEnrichHeatmap(enrichMotifs, n = 20, transpose = TRUE)
tiff("./heatmap_motif.tiff")
ComplexHeatmap::draw(heatmapEM, heatmap_legend_side = "bot", annotation_legend_side = "bot")
dev.off()

In [23]:
# Chromvar ----------------------------------------------------------------
#BiocManager::install("chromVAR", force=TRUE)
#devtools::install_github("GreenleafLab/ArchR", ref="dev_deviationsCPP", repos = BiocManager::repositories())
#to unload a package and reload


if("Motif" %ni% names(proj@peakAnnotation)){
  proj <- addMotifAnnotations(ArchRProj = proj, motifSet = "cisbp", name = "Motif")
}
proj@peakAnnotation$Motif

proj <- addBgdPeaks(proj, force = TRUE)
proj <- addDeviationsMatrix(
  ArchRProj = proj, 
  peakAnnotation = "Motif",
  force = TRUE
)

$Name
[1] "Motif"

$motifs
PWMatrixList of length 884
names(884): Tcfap2a_1 Tcfap2b_2 Tcfap2c_3 ... Mef2b_882 Smad5_883 Smad9_884

$motifSummary
DataFrame with 884 rows and 3 columns
                 name                     ID      strand
          <character>            <character> <character>
Tcfap2a_1     Tcfap2a ENSMUSG00000021359_L..           *
Tcfap2b_2     Tcfap2b ENSMUSG00000025927_L..           *
Tcfap2c_3     Tcfap2c ENSMUSG00000028640_L..           *
Tcfap2e_4     Tcfap2e ENSMUSG00000042477_L..           *
Tcfap2d_5     Tcfap2d ENSMUSG00000042596_L..           *
...               ...                    ...         ...
Foxm1_880       Foxm1 ENSMUSG00000001517_L..           *
Mixl1_881       Mixl1 ENSMUSG00000026497_L..           *
Mef2b_882       Mef2b ENSMUSG00000079033_L..           *
Smad5_883       Smad5 ENSMUSG00000021540_L..           *
Smad9_884       Smad9 ENSMUSG00000027796_L..           *

$Positions
[1] "/home/jinba/workspace/reha/archr_Fhl2/CM_atac/Annotations/M

Identifying Background Peaks!

Using Previous Background Peaks!

ArchR logging to : ArchRLogs/ArchR-addDeviationsMatrix-9a09d1cd65402-Date-2026-03-03_Time-16-04-49.log
If there is an issue, please report to github with logFile!



NULL


2026-03-03 16:04:53 : Batch Execution w/ safelapply!, 0 mins elapsed.

###########
2026-03-03 16:16:57 : Completed Computing Deviations!, 12.141 mins elapsed.
###########

ArchR logging successful to : ArchRLogs/ArchR-addDeviationsMatrix-9a09d1cd65402-Date-2026-03-03_Time-16-04-49.log



In [249]:
plotVarDev <- getVarDeviations(proj, name = "MotifMatrix", plot = TRUE)

DataFrame with 6 rows and 6 columns
     seqnames     idx        name combinedVars combinedMeans      rank
        <Rle> <array>     <array>    <numeric>     <numeric> <integer>
f843        z     843 Smarcc1_843      5.30047    -0.0877069         1
f104        z     104     Fos_104      4.95696    -0.0860027         2
f108        z     108   Bach1_108      4.14100    -0.0748616         3
f640        z     640   Mef2a_640      4.08259    -0.0544142         4
f842        z     842   Mef2d_842      3.48232    -0.0554414         5
f882        z     882   Mef2b_882      3.48232    -0.0554414         6


In [250]:
motifs <- c("Smarcc1", "Fos", "Bach1", "Bach2", "Mef2a", "Jund","Nfatc2","Nfkb","Mef2c","Gata4")
markerMotifs <- getFeatures(proj, select = paste(motifs, collapse="|"), useMatrix = "MotifMatrix")
markerMotifs

[1] "z:Smarcc1_843"          "z:Nfkb1_701"            "z:Nfatc2_700"          
 [4] "z:Nfkb2_699"            "z:Mef2a_640"            "z:Mef2c_638"           
 [7] "z:Gata4_386"            "z:Jund_135"             "z:Bach2_119"           
[10] "z:Fosl2_113"            "z:Bach1_108"            "z:Fosl1_107"           
[13] "z:Fos_104"              "z:Fosb_98"              "deviations:Smarcc1_843"
[16] "deviations:Nfkb1_701"   "deviations:Nfatc2_700"  "deviations:Nfkb2_699"  
[19] "deviations:Mef2a_640"   "deviations:Mef2c_638"   "deviations:Gata4_386"  
[22] "deviations:Jund_135"    "deviations:Bach2_119"   "deviations:Fosl2_113"  
[25] "deviations:Bach1_108"   "deviations:Fosl1_107"   "deviations:Fos_104"    
[28] "deviations:Fosb_98"

In [251]:
proj$Type <- factor(
  proj$Type,
  levels = c("SED", "Ex", "Fhl2OE")
)

In [252]:
getAvailableMatrices(proj)

[1] "GeneIntegrationMatrix" "GeneScoreMatrix"       "MotifMatrix"          
[4] "PeakMatrix"            "TileMatrix"

In [253]:
motifMat <- getMatrixFromProject(proj, "MotifMatrix")
unique(rowData(motifMat)$seqnames)

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-cc2f77c226ec9-Date-2026-04-18_Time-15-52-47.log
If there is an issue, please report to github with logFile!

2026-04-18 15:52:52 : Organizing colData, 0.082 mins elapsed.

2026-04-18 15:52:52 : Organizing rowData, 0.084 mins elapsed.

2026-04-18 15:52:52 : Organizing rowRanges, 0.084 mins elapsed.

2026-04-18 15:52:52 : Organizing Assays (1 of 2), 0.084 mins elapsed.

2026-04-18 15:52:52 : Organizing Assays (2 of 2), 0.088 mins elapsed.

2026-04-18 15:52:52 : Constructing SummarizedExperiment, 0.092 mins elapsed.

2026-04-18 15:52:54 : Finished Matrix Creation, 0.116 mins elapsed.



NULL

In [254]:
motifMat

class: SummarizedExperiment 
dim: 884 9188 
metadata(0):
assays(2): deviations z
rownames(884): Tcfap2a_1 Tcfap2b_2 ... Smad5_883 Smad9_884
rowData names(2): idx name
colnames(9188): Ex2#TTTGCGCTCCATTGAG-1 Ex2#CCGTAGGTCGGCATAT-1 ...
  Fhl2OE2#TTGCAGAGTATTCTCT-1 Fhl2OE2#AGCCTGGGTGTGCGTC-1
colData names(22): BlacklistRatio DoubletEnrichment ... ReadsInPeaks
  FRIP

In [255]:
motifs <- c( "Fos", "Jund","Rela","Gabpa")
markerMotifs <- getFeatures(proj, select = paste(motifs, collapse="|"), useMatrix = "MotifMatrix")
markerMotifs

[1] "z:Rela_698"           "z:Gabpa_273"          "z:Jund_135"          
 [4] "z:Fosl2_113"          "z:Fosl1_107"          "z:Fos_104"           
 [7] "z:Fosb_98"            "deviations:Rela_698"  "deviations:Gabpa_273"
[10] "deviations:Jund_135"  "deviations:Fosl2_113" "deviations:Fosl1_107"
[13] "deviations:Fos_104"   "deviations:Fosb_98"

In [256]:
proj$Type <- as.character(proj$Type)

In [257]:
p <- plotGroups(ArchRProj = proj, 
  groupBy = "Type", 
  colorBy = "MotifMatrix", 
  name = markerMotifs,
  imputeWeights = getImputeWeights(proj)
)

Getting ImputeWeights

Getting Matrix Values...

2026-04-18 15:52:59 : 



ArchR logging to : ArchRLogs/ArchR-imputeMatrix-cc2f72acb947f-Date-2026-04-18_Time-15-53-01.log
If there is an issue, please report to github with logFile!

Using weights on disk

Using weights on disk

1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 




In [258]:
pal = c(
    "Ex"     = "#FF8C00",
    "Fhl2OE" = "#d62728",
    "SED"    = "#2E8B57")

p <- plotGroups(
  ArchRProj = proj, 
  groupBy = "Type", 
  colorBy = "MotifMatrix", 
  name = markerMotifs,
  imputeWeights = getImputeWeights(proj),
  baseSize = 24,
  size = 16,
  pal = pal
)

p

Getting ImputeWeights

Getting Matrix Values...

2026-04-18 15:53:10 : 



ArchR logging to : ArchRLogs/ArchR-imputeMatrix-cc2f7395b9b42-Date-2026-04-18_Time-15-53-13.log
If there is an issue, please report to github with logFile!

Using weights on disk

Using weights on disk

1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 


Picking joint bandwidth of 0.0161



Picking joint bandwidth of 0.0275



Picking joint bandwidth of 0.0586



Picking joint bandwidth of 0.00935



Picking joint bandwidth of 0.0419



...

Picking joint bandwidth of 0.000637



Picking joint bandwidth of 0.00235



Picking joint bandwidth of 0.00308



Picking joint bandwidth of 0.00234



$`z:Rela_698`

$`z:Gabpa_273`

$`z:Jund_135`

$`z:Fosl2_113`

$`z:Fosl1_107`

$`z:Fos_104`

$`z:Fosb_98`

$`deviations:Rela_698`

$`deviations:Gabpa_273`

$`deviations:Jund_135`

$`deviations:Fosl2_113`

$`deviations:Fosl1_107`

$`deviations:Fos_104`

$`deviations:Fosb_98`


In [259]:
getAvailableMatrices(proj)

[1] "GeneIntegrationMatrix" "GeneScoreMatrix"       "MotifMatrix"          
[4] "PeakMatrix"            "TileMatrix"

In [260]:
motifMat <- getMatrixFromProject(proj, useMatrix = "MotifMatrix")
motif_z  <- motifMat[grep("z", rowData(motifMat)$seqnames), ]  # 例：z-scoreのみ抽出
markerMotifs <- getMarkerFeatures(
  ArchRProj  = proj,
  useMatrix  = "MotifMatrix",
  groupBy    = "Type",
  testMethod = "wilcoxon",
  useGroups  = "Ex",
  bgdGroups  = "SED",
  useSeqnames = "z",
  #bias = c("TSSEnrichment", "log10(nFrags)")
)

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-cc2f71453b85f-Date-2026-04-18_Time-15-53-23.log
If there is an issue, please report to github with logFile!

2026-04-18 15:53:27 : Organizing colData, 0.066 mins elapsed.

2026-04-18 15:53:27 : Organizing rowData, 0.068 mins elapsed.

2026-04-18 15:53:27 : Organizing rowRanges, 0.068 mins elapsed.

2026-04-18 15:53:27 : Organizing Assays (1 of 2), 0.068 mins elapsed.

2026-04-18 15:53:27 : Organizing Assays (2 of 2), 0.072 mins elapsed.

2026-04-18 15:53:28 : Constructing SummarizedExperiment, 0.076 mins elapsed.

2026-04-18 15:53:29 : Finished Matrix Creation, 0.097 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-getMarkerFeatures-cc2f74634b0cb-Date-2026-04-18_Time-15-53-29.log
If there is an issue, please report to github with logFile!

MatrixClass = Sparse.Assays.Matrix

2026-04-18 15:53:30 : Matching Known Biases, 0.002 mins elapsed.

2026-04-18 15:53:30 : Computing Pairwise Tests (1 of 1), 0.006 mins elapsed.

Pairwise Test

In [261]:
markerMotifs

class: SummarizedExperiment 
dim: 884 1 
metadata(2): MatchInfo Params
assays(6): Mean FDR ... AUC MeanBGD
rownames(884): 1 2 ... 883 884
rowData names(3): seqnames idx name
colnames(1): Ex
colData names(0):

In [262]:
markerMotifs_Fhl2 <- getMarkerFeatures(
  ArchRProj  = proj,
  useMatrix  = "MotifMatrix",
  groupBy    = "Type",
  testMethod = "wilcoxon",
  useGroups  = "Fhl2OE",
  bgdGroups  = "SED",
  useSeqnames = "z",
  #bias = c("TSSEnrichment", "log10(nFrags)")
)

ArchR logging to : ArchRLogs/ArchR-getMarkerFeatures-cc2f7123e7491-Date-2026-04-18_Time-15-53-31.log
If there is an issue, please report to github with logFile!

MatrixClass = Sparse.Assays.Matrix

2026-04-18 15:53:32 : Matching Known Biases, 0.002 mins elapsed.

2026-04-18 15:53:33 : Computing Pairwise Tests (1 of 1), 0.025 mins elapsed.

Pairwise Test Fhl2OE : Seqnames z

###########
2026-04-18 15:53:35 : Completed Pairwise Tests, 0.051 mins elapsed.
###########

ArchR logging successful to : ArchRLogs/ArchR-getMarkerFeatures-cc2f7123e7491-Date-2026-04-18_Time-15-53-31.log



In [263]:
markerMotifs_Fhl2

class: SummarizedExperiment 
dim: 884 1 
metadata(2): MatchInfo Params
assays(6): Mean FDR ... AUC MeanBGD
rownames(884): 1 2 ... 883 884
rowData names(3): seqnames idx name
colnames(1): Fhl2OE
colData names(0):

In [264]:
ex_up_df <- getMarkers(
  markerMotifs,
  cutOff = "FDR <= 0.05 & MeanDiff >= 0.1"
)$Ex
write.csv(ex_up_df,"ex_up_df.csv")

In [265]:
Fhl2_up_df <- getMarkers(
  markerMotifs_Fhl2,
  cutOff = "FDR <= 0.05 & MeanDiff >= 0.1"
)$Fhl2
write.csv(Fhl2_up_df,"Fhl2_up_df.csv")

In [266]:
ex_up_df

DataFrame with 70 rows and 5 columns
    seqnames     idx          name         FDR  MeanDiff
       <Rle> <array>       <array>   <numeric> <numeric>
146        z     146      Ctcf_146 2.97812e-42  1.444491
820        z     820     Ctcfl_820 2.45780e-39  1.241712
849        z     849       Pgr_849 6.95681e-11  0.507266
278        z     278      Etv4_278 2.29554e-10  0.437482
276        z     276      Etv5_276 8.72270e-10  0.434411
...      ...     ...           ...         ...       ...
269        z     269      Spic_269   0.0413742  0.164942
67         z      67      Atoh8_67   0.0463314  0.184677
879        z     879    Dmrta2_879   0.0479937  0.154898
824        z     824      Rbpj_824   0.0482230  0.179749
825        z     825 Rbpsuhrs3_825   0.0482230  0.179749

In [267]:
rowData(markerMotifs)

DataFrame with 884 rows and 3 columns
    seqnames     idx      name
       <Rle> <array>   <array>
1          z       1 Tcfap2a_1
2          z       2 Tcfap2b_2
3          z       3 Tcfap2c_3
4          z       4 Tcfap2e_4
5          z       5 Tcfap2d_5
...      ...     ...       ...
880        z     880 Foxm1_880
881        z     881 Mixl1_881
882        z     882 Mef2b_882
883        z     883 Smad5_883
884        z     884 Smad9_884

In [268]:
library(ArchR)
library(dplyr)
library(VennDiagram)
library(grid)

###############################################
# 1. Ex の motif（上昇・低下）を取得
###############################################

## Ex 上昇 (FDR<0.05 & MeanDiff>=0.5)
ex_up_df <- getMarkers(
  markerMotifs,
  cutOff = "FDR <= 0.05 & MeanDiff >= 0.25"
)$Ex

ex_up <- ex_up_df$name

## Ex 低下 (FDR<0.05 & MeanDiff<= -0.5)
ex_down_df <- getMarkers(
  markerMotifs,
  cutOff = "FDR <= 0.05 & MeanDiff <= -0.25"
)$Ex

ex_down <- ex_down_df$name


###############################################
# 2. Fhl2 の motif（上昇・低下）を取得
###############################################

## Fhl2 上昇
fhl2_up_df <- getMarkers(
  markerMotifs_Fhl2,
  cutOff = "FDR <= 0.05 & MeanDiff >= 0.25"
)$Fhl2

fhl2_up <- fhl2_up_df$name

## Fhl2 低下
fhl2_down_df <- getMarkers(
  markerMotifs_Fhl2,
  cutOff = "FDR <= 0.05 & MeanDiff <= -0.25"
)$Fhl2

fhl2_down <- fhl2_down_df$name


###############################################
# 3. ベン図（上昇）
###############################################
venn_up <- venn.diagram(
  x = list(
    Ex_up = ex_up,
    Fhl2_up = fhl2_up
  ),
  filename = NULL,
  fill = c("#4DBBD5", "#E64B35"),
  alpha = 0.5,
  cex = 4,
  cat.cex = 1.8,
  main = "Upregulated Motifs (FDR<0.05 & |MeanDiff|>0.25)",
  reverse = TRUE
)

grid.newpage()
grid.draw(venn_up)


###############################################
# 4. ベン図（低下）
###############################################
venn_down <- venn.diagram(
  x = list(
    Ex_down = ex_down,
    Fhl2_down = fhl2_down
  ),
  filename = NULL,
  fill = c("#00A087", "#3C5488"),
  alpha = 0.5,
  cex = 4,
  cat.cex = 1.8,
  main = "Downregulated Motifs (FDR<0.05 & |MeanDiff|>0.25)"
)

grid.newpage()
grid.draw(venn_down)


In [269]:
venn_up <- venn.diagram(
  x = list(
    Ex_up = ex_up,
    Fhl2_up = fhl2_up
  ),
  filename = NULL,
  fill = c("#4DBBD5", "#E64B35"),
  alpha = 0.5,
  cex = 4,
  cat.cex = 0,
  main = "Upregulated Motifs (FDR<0.05 & |MeanDiff|>0.25)",
  main.cex = 2.5,
  fontfamily = "Arial",        # 数字
  cat.fontfamily = "Arial",    # セット名
  main.fontfamily = "Arial"    # タイトル
)

grid.newpage()
grid.draw(venn_up)


Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warnin

In [270]:
venn_down <- venn.diagram(
  x = list(
    Ex_down = ex_down,
    Fhl2_down = fhl2_down
  ),
  filename = NULL,
  fill = c("#00A087", "#3C5488"),
  alpha = 0.5,
  cex = 4,
  cat.cex = 0,
  main = "Downregulated Motifs",
  main.cex = 2.5,
  fontfamily = "Arial",        # 数字
  cat.fontfamily = "Arial",    # セット名
  main.fontfamily = "Arial"    # タイトル
)

grid.newpage()
grid.draw(venn_down)


Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Arial' not found, will use 'sans' instead”
Warnin

In [273]:

###############################################
# 共通上昇 motif
###############################################

common_up_names <- intersect(ex_up_df$name, fhl2_up_df$name)

common_up_table <- merge(
  ex_up_df[, c("name", "FDR", "MeanDiff")],
  fhl2_up_df[, c("name", "FDR", "MeanDiff")],
  by = "name",
  suffixes = c("_Ex", "_Fhl2")
)

common_up_table <- common_up_table[common_up_table$name %in% common_up_names, ]


###############################################
# 共通低下 motif
###############################################

common_down_names <- intersect(ex_down_df$name, fhl2_down_df$name)

common_down_table <- merge(
  ex_down_df[, c("name", "FDR", "MeanDiff")],
  fhl2_down_df[, c("name", "FDR", "MeanDiff")],
  by = "name",
  suffixes = c("_Ex", "_Fhl2")
)

common_down_table <- common_down_table[common_down_table$name %in% common_down_names, ]


###############################################
# 結果表示
###############################################

common_up_table
common_down_table

DataFrame with 29 rows and 5 columns
           name      FDR_Ex MeanDiff_Ex    FDR_Fhl2 MeanDiff_Fhl2
    <character>   <numeric>   <numeric>   <numeric>     <numeric>
1        Ar_687 5.36838e-06    0.301365 1.46109e-11      0.497894
2      Ctcf_146 2.97812e-42    1.444491 2.15623e-55      1.536049
3     Ctcfl_820 2.45780e-39    1.241712 3.81603e-38      1.088672
4      Elf2_286 3.45928e-06    0.326746 2.98471e-09      0.429518
5      Elf4_283 8.31790e-05    0.280074 1.69838e-09      0.436416
...         ...         ...         ...         ...           ...
25   Gm5454_290 9.14984e-10    0.429833 1.02766e-08      0.410988
26    Nr2c2_659 5.65284e-07    0.376956 3.59281e-08      0.410358
27    Nr3c1_673 1.04038e-07    0.416765 4.53716e-84      1.646811
28      Pgr_849 6.95681e-11    0.507266 6.13322e-91      1.800959
29  Smarcc2_646 1.98316e-04    0.268294 4.34539e-12      0.521014

DataFrame with 44 rows and 5 columns
           name      FDR_Ex MeanDiff_Ex    FDR_Fhl2 MeanDiff_Fhl2
    <character>   <numeric>   <numeric>   <numeric>     <numeric>
1      Arntl_89 9.16958e-09   -0.419331 3.07970e-19     -0.621580
2      Atf5_791 8.70022e-04   -0.287300 3.88316e-03     -0.250329
3    Bhlhe40_55 1.11102e-03   -0.261085 1.73031e-15     -0.549554
4    Bhlhe41_56 7.91241e-05   -0.294394 3.09715e-11     -0.457744
5      Clock_52 1.52756e-05   -0.303347 7.84022e-10     -0.412214
...         ...         ...         ...         ...           ...
40      Usf1_46 1.75977e-11   -0.470901 6.13862e-17     -0.611554
41      Usf2_95 3.33782e-09   -0.425976 6.72340e-16     -0.606599
42   Zfp263_878 1.98992e-05   -0.268850 8.03181e-09     -0.317333
43   Zfp281_193 2.05900e-24   -0.688637 1.12281e-10     -0.444026
44   Zfp740_204 2.93036e-14   -0.555815 1.80511e-06     -0.342813

In [274]:
write.csv(common_down_table,"chromvar_common_down_table.csv")

In [275]:
write.csv(common_up_table,"chromvar_common_up_table.csv")

In [276]:
library(dplyr)

## Ex vs SED
ex_df <- getMarkers(markerMotifs, cutOff = "FDR <= 1")$Ex
ex_df <- as.data.frame(ex_df)
ex_df$group <- "Ex"

## Fhl2OE vs SED
fhl2_df <- getMarkers(markerMotifs_Fhl2, cutOff = "FDR <= 1")$Fhl2
fhl2_df <- as.data.frame(fhl2_df)
fhl2_df$group <- "Fhl2OE"


In [277]:
library(ggplot2)

plot_volcano <- function(df, title="Volcano") {

  df$logFDR <- -log10(df$FDR)

  df$sig <- "NS"
  df$sig[df$FDR <= 0.05 & df$MeanDiff >= 0.25]  <- "Up"
  df$sig[df$FDR <= 0.05 & df$MeanDiff <= -0.25] <- "Down"

  ggplot(df, aes(MeanDiff, logFDR)) +
    geom_point(aes(color=sig), size=2, alpha=0.8) +
    scale_color_manual(values=c(
      Up="#E64B35",
      Down="#3C5488",
      NS="grey80"
    )) +
    geom_vline(xintercept=c(-0.25,0.25), linetype=2) +
    geom_hline(yintercept=-log10(0.05), linetype=2) +
    theme_classic(base_size=14) +
    labs(x="MeanDiff", y="-log10(FDR)", title=title)
}


In [279]:
ex_sig <- ex_df %>%
  filter(FDR <= 0.05, abs(MeanDiff) >= 0.25) %>%
  select(name, MeanDiff)

fhl2_sig <- fhl2_df %>%
  filter(FDR <= 0.05, abs(MeanDiff) >= 0.25) %>%
  select(name, MeanDiff)

merged <- inner_join(ex_sig, fhl2_sig, by="name",
                     suffix=c("_Ex","_Fhl2"))

same_dir <- merged %>%
  filter(sign(MeanDiff_Ex)==sign(MeanDiff_Fhl2))

highlight_motifs <- same_dir$name


In [307]:
## 有意のみ
ex_sig <- ex_df %>%
  dplyr::filter(FDR <= 0.05, abs(MeanDiff) >= 0.25) %>%
  dplyr::select(name, MeanDiff)

fhl2_sig <- fhl2_df %>%
  dplyr::filter(FDR <= 0.05, abs(MeanDiff) >= 0.25) %>%
  dplyr::select(name, MeanDiff)

merged <- dplyr::inner_join(
  ex_sig, fhl2_sig, by="name",
  suffix=c("_Ex","_Fhl2")
)

## 同方向のみ
shared_up   <- merged$name[ merged$MeanDiff_Ex >0 & merged$MeanDiff_Fhl2 >0 ]
shared_down <- merged$name[ merged$MeanDiff_Ex <0 & merged$MeanDiff_Fhl2 <0 ]


In [308]:
ex_df$logFDR <- -log10(ex_df$FDR)

ex_df$status <- "NS"

## Ex単独Up/Down
ex_df$status[ex_df$FDR<=0.05 & ex_df$MeanDiff>=0.25]  <- "Ex Up"
ex_df$status[ex_df$FDR<=0.05 & ex_df$MeanDiff<=-0.25] <- "Ex Down"

## 共通は上書き
ex_df$status[ex_df$name %in% shared_up]   <- "SharedUp (Ex and Fhl2OE)"
ex_df$status[ex_df$name %in% shared_down] <- "SharedDown (Ex and Fhl2OE)"


In [309]:
label_keywords <- c(
 #   "Ctcf",
  "Gabpa",
  "Hey",
#  "Nr3c1",
  "Fosl1",
#  "Pbx3",
  "Rela",
  "Junb",
  "Ar_",
    "Id2"
 #   "Nfya"
)
label_idx <- Reduce(
  "|",
  lapply(label_keywords, function(k)
    grepl(k, ex_df$name, ignore.case=FALSE)
  )
)

label_df <- ex_df[label_idx, ]
label_df$label <- sub("_.*", "", label_df$name)

In [315]:
options(repr.plot.width = 10, repr.plot.height = 6)
ggplot(ex_df, aes(MeanDiff, pmin(logFDR, 20))) +

  geom_point(aes(color=status), size=2.3, alpha=0.9) +

  scale_color_manual(values=c(
    NS="grey85",
    "Ex Up"="#F4A6A6",
    "Ex Down"="#A6C8F4",
    "SharedUp (Ex and Fhl2OE)"="#D7301F",
    "SharedDown (Ex and Fhl2OE)"="#084594"
  )) +

  geom_vline(xintercept=c(-0.25,0.25), linetype=2) +
  geom_hline(yintercept=-log10(0.05), linetype=2) +

  geom_text_repel(
    data = label_df,
    aes(y = pmin(logFDR, 20), label=label),
    size=8,
    box.padding=0.35,
    max.overlaps=100
  ) +

  theme_classic(base_size=14) +
  labs(
    title="Motif enrichment: Ex vs SED",
    x="Mean Difference",
    y="-log10(FDR)"
  ) +
  theme(
    legend.title = element_text(size = 20),
    legend.text  = element_text(size = 20),
    legend.key.size = unit(1.5, "cm"),

    axis.title.x = element_text(size = 20, face = "bold"),
    axis.title.y = element_text(size = 20, face = "bold"),

    axis.text.x  = element_text(size = 18),
    axis.text.y  = element_text(size = 18)
  )

In [ ]:
proj <- saveArchRProject(ArchRProj = proj)

saveRDS(proj,"./CM_atac_projReha.rds")

In [248]:
setwd("~/workspace/reha/archr_Fhl2/CM_atac")
proj <- readRDS("./CM_atac_projReha.rds")

In [236]:
proj 


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    



class: ArchRProject 
outputDirectory: /home/jinba/workspace/reha/archr_Fhl2/CM_atac 
samples(6): Ex2 SED2 ... Fhl2OE1 Fhl2OE2
sampleColData names(1): ArrowFiles
cellColData names(22): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 9188
medianTSS(1): 8.8045
medianFrags(1): 21582.5